In [1]:
import numpy as np
import pandas as pd
from scipy.special import gammaln
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Load the economic loss data
econ_loss_raw = pd.read_excel("../../data/raw/Economic damages source review.xlsx", sheet_name="Updated numbers")

# Preprocess data
econ_loss = econ_loss_raw.rename(columns={'Fraction GDP losses': 'share_gdp_loss',
                                         'Mortality (SMU)': 'mortality_smu',
                                         'Disease': 'disease'})
econ_loss['pct_gdp_loss'] = econ_loss['share_gdp_loss'] * 100
econ_loss = econ_loss[['share_gdp_loss', 'pct_gdp_loss', 'mortality_smu', 'disease']]
econ_loss_clean = econ_loss.dropna(axis=0)

# Prepare data for Poisson regression
X = np.log(econ_loss_clean[['mortality_smu']])
y = econ_loss_clean['share_gdp_loss']
X_sm = sm.add_constant(X)

# Fit Poisson model
pm = sm.GLM(y, X_sm, family=sm.families.Poisson())
pm_results = pm.fit(cov_type='HC3')
y_pred = pm_results.predict(X_sm)

# Define two versions of McFadden's R2
def mcf_pseudo_r2_with_gamma(y, y_pred):
    """McFadden's R2 including gamma log term"""
    ll_model = np.sum(y * np.log(y_pred) - y_pred - gammaln(y + 1))
    mean_response = np.mean(y)
    ll_null = np.sum(y * np.log(mean_response) - mean_response - gammaln(y + 1))
    return 1 - (ll_model / ll_null)

def mcf_pseudo_r2_no_gamma(y, y_pred):
    """McFadden's R2 without gamma log term"""
    ll_model = np.sum(y * np.log(y_pred) - y_pred)
    mean_response = np.mean(y)
    ll_null = np.sum(y * np.log(mean_response) - mean_response)
    return 1 - (ll_model / ll_null)

# Calculate both versions
r2_with_gamma = mcf_pseudo_r2_with_gamma(y, y_pred)
r2_no_gamma = mcf_pseudo_r2_no_gamma(y, y_pred)
r2_statsmodels = pm_results.pseudo_rsquared(kind='mcf')

print("McFadden's R² Comparison:")
print(f"With gamma term: {r2_with_gamma:.4f}")
print(f"Without gamma term: {r2_no_gamma:.4f}")
print(f"Statsmodels implementation: {r2_statsmodels:.4f}")


McFadden's R² Comparison:
With gamma term: 0.1255
Without gamma term: 0.1111
Statsmodels implementation: 0.1255
